# שיעור 25 — מטריצת בלבול: מי סווג כמי?
45 דקות מוצעות (שעת לימוד אחת), אחרי שיעור 24.

**פתיחה:** דודו בודק מסווג שמסמן מוצרים פגומים. האם מספר התחזיות הנכונות לבדו מגלה כמה פגומים הוחמצו וכמה מוצרים תקינים נשלחו לבדיקה מיותרת?
נלמד לארגן זוגות אמת וחיזוי לטבלת ספירות. אחר כך נקרא את אותה טבלה עבור שלושת מיני Penguins, בלי לשנות את המודל.

עוגן: [סיכום מפגש השתלמות 6](../../08-Classification-Metrics/Meeting-06-Didactic-Summary.md) ו[המחברת המודרכת](../../08-Classification-Metrics/Meeting-06-Classification-Metrics-Guided.ipynb). זוהי התאמה דידקטית חדשה: דוגמת מוצרים מומצאת והמשך Penguins, במקום דוגמאות הבריאות והחתולים. המקורות נשמרו; אין כאן שחזור תרגיל מקור.

## 1. לפני הספרייה — סופרים זוגות
הנתונים הבאים מומצאים. ציירו ביד טבלה בת שתי שורות ושתי עמודות וספרו כל זוג. הסבירו שתי טעויות שונות במילים.

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import matplotlib.pyplot as plt

In [ ]:
# נתונים מומצאים לצורך למידה; אין CSV או מודל מאומן בדוגמה הזאת
actual = ["OK", "OK", "OK", "OK", "Defect", "Defect", "Defect", "Defect"]
predicted = ["OK", "OK", "Defect", "OK", "OK", "OK", "Defect", "Defect"]
pairs = pd.DataFrame({"actual": actual, "predicted": predicted})
display(pairs)

## מוסכמה שחייבים לקרוא
**שורות = אמת (Actual), עמודות = חיזוי (Predicted)**. במטריצת ספירות כל רשומה נספרת בתא אחד. סכום השורה מספר כמה רשומות אמיתיות יש במחלקה; סכום העמודה מספר כמה חיזויים ניתנו למחלקה.
labels קובע במפורש את סדר המחלקות בשני הצירים. אין לקבוע את משמעות התאים לפי מיקום בלבד ללא קריאת תוויות. הסדר כאן מתחיל בשלילי OK; בחומר אחר החיובי עשוי להופיע ראשון.
האלכסון מכיל סיווגים נכונים; מחוץ לאלכסון נמצאות טעויות. הצבע מייצג ספירה, לא רמת ביטחון.

In [ ]:
labels_binary = ["OK", "Defect"]
cm_binary = confusion_matrix(actual, predicted, labels=labels_binary)
table_binary = pd.DataFrame(
    cm_binary, index=labels_binary, columns=labels_binary
)
table_binary.index.name = "Actual"
table_binary.columns.name = "Predicted"
display(table_binary)

## ארבע תוצאות — לאחר שמגדירים חיובי
המחלקה החיובית היא **Defect**, כלומר "דורש טיפול". חיובי אינו אומר "טוב".
| אמת | חיזוי | שם | משמעות |
|---|---|---|---|
| OK | OK | TN | תקין שזוהה כתקין |
| OK | Defect | FP | אזעקת שווא |
| Defect | OK | FN | פגם שהוחמץ |
| Defect | Defect | TP | פגם שזוהה |

חשבו קודם מתוך הזוגות, ואז השוו לקוד. loc הוא קיצור קריאה לפי תווית שורה ותווית עמודה: table.loc[אמת, חיזוי]. הוא מובא כאן לקריאת תאים; אפשר לחשב גם ביד.
$$Accuracy=\frac{TN+TP}{N}$$
N הוא סכום כל תאי המטריצה. אין לחשב ממוצע פשוט של אחוז ההצלחה בכל שורה ולקרוא לו Accuracy.

In [ ]:
tn = table_binary.loc["OK", "OK"]
fp = table_binary.loc["OK", "Defect"]
fn = table_binary.loc["Defect", "OK"]
tp = table_binary.loc["Defect", "Defect"]
n = cm_binary.sum()
accuracy_from_matrix = (tn + tp) / n
print("TN:", tn, "FP:", fp, "FN:", fn, "TP:", tp)
print("Total:", n, "Accuracy:", accuracy_from_matrix)
print("Accuracy from pairs:", accuracy_score(actual, predicted))

**פירוש:** יש 5 הצלחות מתוך 8, כלומר 62.5%. שני פגמים הוחמצו ומוצר תקין אחד סומן כפגום. ציון אחד מאחד טעויות שיש להן משמעות שונה.
אם נגדיר חיובי כ־OK, שמות ארבעת המצבים ישתנו; זוגות האמת והחיזוי וה־Accuracy לא ישתנו.

## ממשיכים את אותו מודל משיעור 24
התא הבא הוא **קוד הכנה נתון שכבר נלמד**, להרצה מלמעלה; אין צורך לכתוב אותו מחדש בשיעור זה.
הוא משחזר בדיוק את המודרכת: סנפיר ומסה, חלוקה 25%, seed=42, stratify, השלמה ותקנון מאימון בלבד ו־K=3 קבוע. כעת מוסיפים דרך תיאור לתחזיות שכבר התקבלו; לא בוחרים פרמטרים לפי Test.
Penguins: עותק הקורס בן 344 שורות ו־7 עמודות. מקור: Kristen Gorman ו־Palmer Station LTER; ייחוס ותיעוד [palmerpenguins](https://github.com/allisonhorst/palmerpenguins), Horst, Hill & Gorman (2020), DOI:10.5281/zenodo.3960218. התאמה לפרויקט גמר טעונה בדיקה נפרדת.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
display(df.head())
display(df.isna().sum())

model_df = df.dropna(subset=["species"]).copy()
features = ["flipper_length_mm","body_mass_g"]
X = model_df[features].copy()
y = model_df["species"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Train:", len(X_train), "Test:", len(X_test))

training_means = X_train.mean()
X_train_ready = X_train.fillna(training_means)
X_test_ready = X_test.fillna(training_means)

scaler = StandardScaler().set_output(transform="pandas")
scaler.fit(X_train_ready)
X_train_scaled = scaler.transform(X_train_ready)
X_test_scaled = scaler.transform(X_test_ready)
display(X_train_scaled.head())

model = KNeighborsClassifier(n_neighbors=3, weights="uniform", metric="euclidean")
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)
print(predictions[:5])

comparison = pd.DataFrame(
    {"actual": y_test, "predicted": predictions}, index=y_test.index
)
comparison["correct"] = comparison["actual"] == comparison["predicted"]
display(comparison.head(10))
correct_count = comparison["correct"].sum()
accuracy = accuracy_score(y_test, predictions)
print("Correct:", correct_count, "out of", len(y_test))
print("Accuracy:", accuracy, "Percent:", 100 * accuracy)

## 2. משלוש מחלקות לטבלה
כל מין מקבל שורה ועמודה. אין "חיובי" יחיד בלי לבחור מחלקת מטרה מול כל היתר. היום נקרא תאים, סכומי שורות ואלכסון; מדדים לכל מחלקה בשיעור הבא.
לפני הרצה: מה מבטא התא בשורת Chinstrap ובעמודת Adelie? האם התא ההפוך אומר אותו דבר?

In [ ]:
labels_species = ["Adelie", "Chinstrap", "Gentoo"]
cm_species = confusion_matrix(y_test, predictions, labels=labels_species)
table_species = pd.DataFrame(
    cm_species, index=labels_species, columns=labels_species
)
table_species.index.name = "Actual"
table_species.columns.name = "Predicted"
display(table_species)

# אלכסון ראשי: החיזוי זהה לאמת
correct_from_matrix = (
    table_species.loc["Adelie", "Adelie"]
    + table_species.loc["Chinstrap", "Chinstrap"]
    + table_species.loc["Gentoo", "Gentoo"]
)
print("N:", cm_species.sum())
print("Accuracy from diagonal:", correct_from_matrix / cm_species.sum())
print("Actual Chinstrap:", table_species.loc["Chinstrap"].sum())
print("Predicted Chinstrap:", table_species["Chinstrap"].sum())
print("Actual Chinstrap predicted Adelie:", table_species.loc["Chinstrap", "Adelie"])

chart = ConfusionMatrixDisplay(
    confusion_matrix=cm_species, display_labels=labels_species
)
chart.plot(cmap="Blues", values_format="d", colorbar=False)
plt.title("Penguins — fixed K=3; test split from lesson 24")
plt.tight_layout()
plt.show()

## 3. תיעוד לפרויקט וכרטיס יציאה
בחרו תא שגיאה שאינו אפס ותארו: כמה דוגמאות ממין אמת מסוים סווגו כמין אחר? אם אין טעויות, ציינו זאת.
דווחו את סדר התוויות, משמעות הצירים, מספר הנבדקים וסכום האלכסון. סכום כל התאים צריך להתאים למספר זוגות הבדיקה.
המטריצה מתארת את החלוקה המסוימת; אי אפשר להכריז ממנה לבדה על התאמה לאוכלוסייה חדשה. גם ספירה קטנה של טעויות במחלקה נדירה אינה מוכיחה ביצוע טוב ללא ידיעת גודלה.
**כרטיס יציאה:** מה ההבדל בין סכום שורת Chinstrap לסכום עמודת Chinstrap? איזו טעות היא FN כאשר החיובי הוא מוצר פגום?